In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pio.renderers.default = "vscode"
print("✅ Imports done")

✅ Imports done


In [2]:
ROOT     = Path(r"C:\Users\sayan\Desktop\Data_Analysis\ai_sustainability_project")
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "charts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILE1 = DATA_DIR / "AI_DataCenter_Sustainability_Datasets.xlsx"
FILE2 = DATA_DIR / "AI_Boom_vs_DC_Sustainability_FINAL.xlsx"

# Consistent colour palette across all charts
PALETTE = {
    'cyan':      '#00B4D8',
    'blue':      '#1D4ED8',
    'green':     '#10B981',
    'red':       '#EF4444',
    'orange':    '#F97316',
    'purple':    '#8B5CF6',
    'yellow':    '#F59E0B',
    'grey':      '#8B949E',
    'google':    '#4285F4',
    'microsoft': '#00A4EF',
    'amazon':    '#FF9900',
    'meta':      '#0866FF',
}

# Dark theme template applied to every chart
DARK_TEMPLATE = {
    'paper_bgcolor': '#0D1117',
    'plot_bgcolor':  '#161B22',
    'font':          dict(color='#E6EDF3', family='Arial'),
    'legend':        dict(bgcolor='#161B22', bordercolor='#30363D'),
}

def make_fig(**kwargs):
    fig = go.Figure(**kwargs)
    fig.update_layout(**DARK_TEMPLATE)
    return fig

print("✅ Paths and config ready")

✅ Paths and config ready


In [20]:
def load_sheet(filepath, sheet_index, header_row=3):
    df = pd.read_excel(filepath, sheet_name=sheet_index, header=header_row)
    df.columns = (df.columns.astype(str)
                  .str.replace(r'\n', ' ', regex=True)
                  .str.strip()
                  .str.replace(r'\s+', ' ', regex=True))
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    return df

def drop_note_rows(df, col_index=0):
    col = df.columns[col_index]
    return df[~df[col].astype(str).str.startswith('📝')].reset_index(drop=True)

def force_numeric(df, exclude=None):
    exclude = exclude or []
    for col in df.columns:
        if col not in exclude:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# ── Load all sheets ──────────────────────────────────────────────────────────
df_dc_global    = drop_note_rows(force_numeric(load_sheet(FILE1, 1), exclude=['Year']))
df_pue_wue      = drop_note_rows(force_numeric(load_sheet(FILE1, 3), exclude=['Year']))
df_dc_carbon    = drop_note_rows(force_numeric(load_sheet(FILE1, 4), exclude=['Year']))

df_ai_models    = drop_note_rows(force_numeric(load_sheet(FILE2, 1),
                    exclude=['Model','Company','Hardware Used','Key Source','Notes']))
df_core_ts      = drop_note_rows(force_numeric(load_sheet(FILE2, 2),
                    exclude=['Data Flag (H=Historical P=Projected)']))
df_hyperscalers = drop_note_rows(force_numeric(load_sheet(FILE2, 3),
                    exclude=['Company','Net Zero Target Year']))
df_ai_kpis      = drop_note_rows(force_numeric(load_sheet(FILE2, 4),
                    exclude=['Data Flag (H/P)']))
df_dc_sust      = drop_note_rows(force_numeric(load_sheet(FILE2, 5), exclude=['Year']))
df_inference    = drop_note_rows(force_numeric(load_sheet(FILE2, 6),
                    exclude=['AI Service / Model','Notes']))
df_capex_gap    = drop_note_rows(force_numeric(load_sheet(FILE2, 8, header_row=4),
                    exclude=['Company','Net Zero Target','Assessment']))

# ── Split historical vs projected ────────────────────────────────────────────
flag_col  = [c for c in df_core_ts.columns if 'Flag' in c][0]
df_hist   = df_core_ts[df_core_ts[flag_col].astype(str).str.strip() == 'H'].copy()
df_proj   = df_core_ts[df_core_ts[flag_col].astype(str).str.strip() == 'P'].copy()

kpis_flag = [c for c in df_ai_kpis.columns if 'Flag' in c or 'flag' in c][0]
df_kpis_hist = df_ai_kpis[df_ai_kpis[kpis_flag].astype(str).str.strip() == 'H'].copy()

print(f"✅ All sheets loaded")
print(f"   Historical rows : {len(df_hist)}")
print(f"   Projected rows  : {len(df_proj)}")

✅ All sheets loaded
   Historical rows : 10
   Projected rows  : 5


In [21]:
print(df_capex_gap.columns.tolist())
print(df_capex_gap.head(3))

['Company', 'Year', 'DC Capex ($B)', 'Total GHG (MtCO₂e)', 'GHG vs 2020 Base (%chg)', 'Pledged Annual Reduction (%)', 'Actual Annual Change (%)', 'Gap (Pledge-Actual %)', 'Renewable PPA (GW)', 'Scope 3 as % of Total GHG', 'Sustainability Score (1–5)', 'Net Zero Target', 'Assessment']
  Company    Year  DC Capex ($B)  Total GHG (MtCO₂e)  GHG vs 2020 Base (%chg)  \
0  Google  2019.0           10.0                 9.7                      NaN   
1  Google  2020.0           12.0                 NaN                      0.0   
2  Google  2021.0           15.0                10.6                      9.3   

   Pledged Annual Reduction (%)  Actual Annual Change (%)  \
0                          -6.7                       NaN   
1                          -6.7                       0.0   
2                          -6.7                       9.3   

   Gap (Pledge-Actual %)  Renewable PPA (GW)  Scope 3 as % of Total GHG  \
0                    NaN                 3.5                       93.

In [5]:
# ── What this proves ─────────────────────────────────────────────────────────
# DC energy grew 114% from 2015 to 2024. The rate of growth visibly
# accelerates after 2022 — the AI inflection point. This is the opening
# chart of your report — it establishes the scale of the problem immediately.

yr  = 'Year'
twh = 'Global DC Energy (TWh)'

df_full = pd.concat([df_hist, df_proj]).dropna(subset=[yr, twh])
df_full[yr] = df_full[yr].astype(int)

hist = df_full[df_full[flag_col].str.strip() == 'H']
proj = df_full[df_full[flag_col].str.strip() == 'P']

# Bridge point connects last historical to first projected smoothly
bridge_x = [hist[yr].iloc[-1], proj[yr].iloc[0]]
bridge_y = [hist[twh].iloc[-1], proj[twh].iloc[0]]

fig = make_fig()

# Shaded area under historical curve
fig.add_trace(go.Scatter(
    x=hist[yr], y=hist[twh],
    fill='tozeroy', fillcolor='rgba(0,180,216,0.08)',
    line=dict(color=PALETTE['cyan'], width=3),
    mode='lines+markers',
    marker=dict(size=7, color=PALETTE['cyan']),
    name='Historical (IEA verified)'
))

# Bridge to projection
fig.add_trace(go.Scatter(
    x=bridge_x, y=bridge_y,
    line=dict(color=PALETTE['cyan'], width=2.5, dash='dash'),
    mode='lines', showlegend=False
))

# Projected line
fig.add_trace(go.Scatter(
    x=proj[yr], y=proj[twh],
    fill='tozeroy', fillcolor='rgba(0,180,216,0.04)',
    line=dict(color=PALETTE['cyan'], width=2.5, dash='dash'),
    mode='lines+markers',
    marker=dict(size=7, color=PALETTE['cyan'], symbol='circle-open'),
    name='Projected — IEA Base Case'
))

# Key event annotations
events = [
    (2020, 245, "COVID-19 Cloud Surge"),
    (2022, 310, "ChatGPT Launched"),
    (2023, 370, "GPT-4 Released"),
    (2024, 415, "AI Capex Supercycle"),
]
for ev_yr, ev_twh, ev_label in events:
    fig.add_annotation(
        x=ev_yr, y=ev_twh,
        text=ev_label,
        showarrow=True,
        arrowhead=2,
        arrowcolor=PALETTE['yellow'],
        arrowwidth=1.5,
        ax=0, ay=-45,
        font=dict(size=10, color=PALETTE['yellow']),
        bgcolor='#21262D',
        bordercolor=PALETTE['yellow'],
        borderpad=4,
    )

# Forecast divider line
fig.add_vline(x=2024.5, line_dash='dot', line_color='#30363D', line_width=1.5)
fig.add_annotation(x=2025, y=980, text="← Forecast",
                   showarrow=False, font=dict(size=9, color='#8B949E'))

fig.update_layout(
    title=dict(
        text='Global Data Center Electricity Consumption (2015–2030)',
        font=dict(size=18, color='#E6EDF3'), x=0.5
    ),
    xaxis=dict(title='Year', gridcolor='#21262D', dtick=1),
    yaxis=dict(title='Electricity Consumption (TWh)', gridcolor='#21262D'),
    legend=dict(x=0.02, y=0.98),
    height=520,
    annotations=fig.layout.annotations + (
        go.layout.Annotation(
            text='Source: IEA Energy & AI Report 2025 | LBNL 2024',
            x=0, y=-0.12, xref='paper', yref='paper',
            showarrow=False, font=dict(size=9, color='#8B949E')
        ),
    )
)

fig.write_html(str(OUT_DIR / "02_chart1_dc_energy_hero.html"))
fig.show()
print("✅ Chart 1 saved")

✅ Chart 1 saved


In [6]:
# ── What this proves ─────────────────────────────────────────────────────────
# AI investment and DC energy move together. Before running any correlation
# numbers, this chart makes the relationship visually undeniable.
# Two y-axes: DC energy (TWh) left, AI investment ($B) right.

inv_col   = [c for c in df_kpis_hist.columns if 'US Private' in c or 'Private AI' in c][0]
genai_col = [c for c in df_kpis_hist.columns if 'GenAI' in c and 'Inv' in c][0]

years_common = sorted(
    set(df_hist['Year'].astype(int)) & set(df_kpis_hist['Year'].astype(int))
)

dc_e    = df_hist.set_index(df_hist['Year'].astype(int))[twh].reindex(years_common)
ai_inv  = df_kpis_hist.set_index('Year')[inv_col].reindex(years_common)
genai_i = df_kpis_hist.set_index('Year')[genai_col].reindex(years_common)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.update_layout(**DARK_TEMPLATE)

fig.add_trace(go.Scatter(
    x=years_common, y=dc_e.values,
    name='Global DC Energy (TWh)',
    line=dict(color=PALETTE['cyan'], width=3),
    mode='lines+markers', marker=dict(size=7),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=years_common, y=ai_inv.values,
    name='US AI Private Investment ($B)',
    line=dict(color=PALETTE['orange'], width=2.5, dash='dash'),
    mode='lines+markers', marker=dict(size=7, symbol='square'),
), secondary_y=True)

fig.add_trace(go.Scatter(
    x=years_common, y=genai_i.values,
    name='GenAI Investment ($B)',
    line=dict(color=PALETTE['purple'], width=2, dash='dot'),
    mode='lines+markers', marker=dict(size=6, symbol='triangle-up'),
), secondary_y=True)

fig.add_vrect(
    x0=2022, x1=2024,
    fillcolor=PALETTE['yellow'], opacity=0.06,
    layer='below', line_width=0,
    annotation_text='AI Inflection Zone',
    annotation_position='top left',
    annotation_font=dict(color=PALETTE['yellow'], size=10)
)

fig.update_layout(
    title=dict(
        text='The AI Boom & Data Center Energy Demand Move Together',
        font=dict(size=17, color='#E6EDF3'), x=0.5
    ),
    height=520,
    legend=dict(x=0.02, y=0.98),
)
fig.update_yaxes(title_text='DC Electricity Consumption (TWh)',
                 gridcolor='#21262D', secondary_y=False)
fig.update_yaxes(title_text='AI Investment ($B)',
                 gridcolor='#21262D', secondary_y=True)
fig.update_xaxes(title_text='Year', dtick=1, gridcolor='#21262D')

fig.write_html(str(OUT_DIR / "02_chart2_ai_boom_overlay.html"))
fig.show()
print("✅ Chart 2 saved")

✅ Chart 2 saved


In [11]:
# ── What this proves ─────────────────────────────────────────────────────────
# Building bigger AI models costs exponentially more energy.
# We fit a power-law on log-log axes. The slope tells you exactly
# how much extra energy each doubling of model size costs.

models_clean = df_ai_models.dropna(
    subset=['Parameters (Billions)', 'Training Energy (MWh)']
).copy()
models_clean = models_clean[
    models_clean['Model'].astype(str) != 'nan'
].copy()

params  = models_clean['Parameters (Billions)'].values.astype(float)
energy  = models_clean['Training Energy (MWh)'].values.astype(float)
names   = models_clean['Model'].values
years_m = pd.to_numeric(models_clean['Release Year'], errors='coerce').values

# Power-law fit in log space
slope, intercept, r_value, _, _ = stats.linregress(
    np.log10(params), np.log10(energy)
)
x_fit = np.logspace(np.log10(params.min()), np.log10(params.max()), 100)
y_fit = 10 ** (slope * np.log10(x_fit) + intercept)

fig = make_fig()

# Trendline first so dots render on top
fig.add_trace(go.Scatter(
    x=x_fit, y=y_fit,
    mode='lines',
    line=dict(color=PALETTE['yellow'], width=2, dash='dash'),
    name=f'Power-law fit: slope = {slope:.2f}',
))

# Data points coloured by year
fig.add_trace(go.Scatter(
    x=params, y=energy,
    mode='markers+text',
    marker=dict(
        size=12,
        color=years_m,
        colorscale='Plasma',
        colorbar=dict(title='Release Year', thickness=12),
        line=dict(color='#30363D', width=1),
    ),
    text=names,
    textposition='top right',
    textfont=dict(size=9, color='#E6EDF3'),
    name='AI Models',
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Parameters: %{x:.1f}B<br>'
        'Training Energy: %{y:,.0f} MWh<br>'
        '<extra></extra>'
    )
))

fig.update_layout(
    title=dict(
        text='Compute Scaling Law: Bigger AI Models = Exponentially More Energy',
        font=dict(size=16, color='#E6EDF3'), x=0.5
    ),
    xaxis=dict(type='log', title='Model Parameters (Billions) — log scale',
               gridcolor='#21262D'),
    yaxis=dict(type='log', title='Training Energy (MWh) — log scale',
               gridcolor='#21262D'),
    height=560,
    legend=dict(x=0.02, y=0.02),  # bottom left — away from colorbar
    annotations=[
        go.layout.Annotation(
            x=0.55, y=0.12,  # moved down and right of centre — below the data cluster
            xref='paper', yref='paper',
            text=f'R² = {r_value**2:.3f}<br>Doubling params → ×{2**slope:.2f} energy',
            showarrow=False,
            font=dict(size=11, color=PALETTE['yellow']),
            bgcolor='#21262D',
            bordercolor=PALETTE['yellow'],
            borderpad=6, align='left'
        ),
        go.layout.Annotation(
            text='Source: Epoch AI | Patterson et al. 2021 | arXiv | Stanford HAI 2025',
            x=0, y=-0.12, xref='paper', yref='paper',
            showarrow=False, font=dict(size=9, color='#8B949E')
        ),
    ]
)

fig.write_html(str(OUT_DIR / "02_chart3_compute_scaling_law.html"))
fig.show()

print(f"✅ Chart 3 saved")
print(f"   Power-law slope  : {slope:.3f}")
print(f"   R²               : {r_value**2:.3f}")
print(f"   Doubling params → ×{2**slope:.2f} training energy")

✅ Chart 3 saved
   Power-law slope  : 0.861
   R²               : 0.885
   Doubling params → ×1.82 training energy


In [13]:
# ── What this proves ─────────────────────────────────────────────────────────
# Inference is the hidden sustainability problem. Training gets all the
# attention but running these models daily at scale costs far more energy
# over a model's lifetime. The gap between Google Search and DeepSeek-R1
# is the single most striking number in the project.

inf_clean = df_inference.dropna(
    subset=['AI Service / Model', 'Energy per Query (Wh)']
).copy()
inf_clean = inf_clean[
    ~inf_clean['AI Service / Model'].astype(str).str.contains(
        'ALL|📝|nan', na=True)
].copy()
inf_clean = inf_clean.sort_values('Energy per Query (Wh)', ascending=True)

services = inf_clean['AI Service / Model'].astype(str).values
energy_q = inf_clean['Energy per Query (Wh)'].values.astype(float)

def get_color(s):
    if 'Google Search' in s:                              return PALETTE['green']
    if any(x in s for x in ['R1', 'o3']):                return PALETTE['red']
    if any(x in s for x in ['mini', 'Phi', 'GPT-3.5',
                             'Gemma', 'Mistral']):        return PALETTE['cyan']
    return PALETTE['blue']

colors = [get_color(s) for s in services]

fig = make_fig()

fig.add_trace(go.Bar(
    x=energy_q, y=services,
    orientation='h',
    marker=dict(color=colors, line=dict(color='#21262D', width=0.5)),
    text=[f'{v:.4g} Wh' for v in energy_q],
    textposition='outside',
    textfont=dict(size=9, color='#E6EDF3'),
    hovertemplate='<b>%{y}</b><br>Energy per query: %{x:.4g} Wh<extra></extra>',
    name=''
))

# Google Search baseline
search_val = inf_clean[
    inf_clean['AI Service / Model'].astype(str).str.contains('Google Search')
]['Energy per Query (Wh)'].values

if len(search_val):
    fig.add_vline(
        x=search_val[0],
        line_dash='dot',
        line_color=PALETTE['green'],
        line_width=1.5,
        annotation_text='Google Search baseline',
        annotation_font=dict(color=PALETTE['green'], size=9),
        annotation_position='top right'
    )

# Ratio callout
r1_val = inf_clean[
    inf_clean['AI Service / Model'].astype(str).str.contains('DeepSeek-R1')
]['Energy per Query (Wh)'].values

if len(search_val) and len(r1_val):
    ratio = r1_val[0] / search_val[0]
    fig.add_annotation(
        x=r1_val[0] * 0.3,
        y=len(services) - 1,
        text=f'<b>{ratio:,.0f}×</b> more energy<br>than a Google Search',
        showarrow=True, arrowhead=2,
        arrowcolor=PALETTE['red'],
        ax=80, ay=40,
        font=dict(size=11, color=PALETTE['red']),
        bgcolor='#21262D',
        bordercolor=PALETTE['red'], borderpad=5
    )
    print(f"   DeepSeek-R1 vs Google Search: {ratio:,.0f}× more energy per query")

fig.update_layout(
    title=dict(
        text='The Hidden Cost of AI: Energy Per Query by Service',
        font=dict(size=16, color='#E6EDF3'), x=0.5
    ),
    xaxis=dict(
        type='log',
        title='Energy per Query (Wh) — log scale',
        gridcolor='#21262D'
    ),
    yaxis=dict(title='', gridcolor='#21262D'),
    height=580,
    showlegend=False,
    margin=dict(l=200),
)

fig.write_html(str(OUT_DIR / "02_chart4_inference_energy.html"))
fig.show()
print("✅ Chart 4 saved")

   DeepSeek-R1 vs Google Search: 110,000× more energy per query


✅ Chart 4 saved


In [16]:
# ── What this proves ─────────────────────────────────────────────────────────
# Every major hyperscaler claims 100% renewable energy.
# Yet all four companies show rising absolute GHG emissions year on year.
# This is the central contradiction of Act II — and it's in their own data.

companies   = ['Google', 'Microsoft', 'Amazon', 'Meta']
comp_colors = [PALETTE['google'], PALETTE['microsoft'],
               PALETTE['amazon'], PALETTE['meta']]

ghg_col   = [c for c in df_hyperscalers.columns if 'Total GHG' in c][0]
ren_col_h = [c for c in df_hyperscalers.columns if 'Renewable' in c and '%' in c][0]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.update_layout(**DARK_TEMPLATE)

for company, color in zip(companies, comp_colors):
    cdf = (df_hyperscalers[df_hyperscalers['Company'] == company]
           .dropna(subset=['Year', ghg_col])
           .copy())
    cdf['Year']  = cdf['Year'].astype(int)
    cdf[ghg_col] = pd.to_numeric(cdf[ghg_col], errors='coerce')
    cdf          = cdf.sort_values('Year')

    fig.add_trace(go.Scatter(
        x=cdf['Year'], y=cdf[ghg_col],
        name=company,
        line=dict(color=color, width=2.5),
        mode='lines+markers',
        marker=dict(size=8),
        hovertemplate=(
            f'<b>{company}</b><br>'
            'Year: %{x}<br>'
            'GHG: %{y:.1f} MtCO₂e<extra></extra>'
        )
    ), secondary_y=False)

# Renewable % using Google as the representative line
g_df = (df_hyperscalers[df_hyperscalers['Company'] == 'Google']
        .dropna(subset=['Year', ren_col_h]).copy())
g_df['Year']     = g_df['Year'].astype(int)
g_df[ren_col_h]  = pd.to_numeric(g_df[ren_col_h], errors='coerce')
g_df             = g_df.sort_values('Year')

fig.add_trace(go.Scatter(
    x=g_df['Year'], y=g_df[ren_col_h],
    name='Renewable Energy % (all 4 hit 100%)',
    line=dict(color=PALETTE['green'], width=2, dash='dot'),
    mode='lines+markers',
    marker=dict(size=6, symbol='triangle-up'),
    hovertemplate='Renewable %%: %{y:.0f}%%<extra></extra>'
), secondary_y=True)

# 100% threshold line
fig.add_hline(
    y=100, secondary_y=True,
    line_dash='dash',
    line_color=PALETTE['green'],
    line_width=1,
    opacity=0.5,
    annotation_text='100% Renewable threshold',
    annotation_font=dict(color=PALETTE['green'], size=9)
)

fig.update_layout(
    title=dict(
        text='The Hyperscaler Paradox: 100% Renewable Energy → Still Rising Emissions',
        font=dict(size=16, color='#E6EDF3'), x=0.5
    ),
    height=540,
    legend=dict(
    x=1.08,       # pushes legend to the right of the chart
    y=1.0,
    xanchor='left',
    yanchor='top'
),
    annotations=[
        go.layout.Annotation(
            x=2022, y=18,
            text='All 4 hit 100% renewable<br>yet emissions kept rising',
            showarrow=True, arrowhead=2,
            arrowcolor=PALETTE['yellow'],
            ax=-80, ay=-40,
            font=dict(size=10, color=PALETTE['yellow']),
            bgcolor='#21262D',
            bordercolor=PALETTE['yellow'], borderpad=5
        ),
        go.layout.Annotation(
            text='Source: Company Sustainability Reports 2019–2024 | ITU/WBA 2024',
            x=0, y=-0.12, xref='paper', yref='paper',
            showarrow=False, font=dict(size=9, color='#8B949E')
        ),
    ]
)

fig.update_yaxes(
    title_text='Total GHG Emissions (MtCO₂e)',
    gridcolor='#21262D', secondary_y=False
)
fig.update_yaxes(
    title_text='Renewable Energy (%)',
    gridcolor='#21262D',
    secondary_y=True,
    range=[0, 130]
)
fig.update_xaxes(title_text='Year', dtick=1, gridcolor='#21262D')

fig.write_html(str(OUT_DIR / "02_chart5_hyperscaler_paradox.html"))
fig.show()
print("✅ Chart 5 saved")

✅ Chart 5 saved


In [22]:
# ── What this proves ─────────────────────────────────────────────────────────
# The gap column = pledged annual reduction % minus actual annual change %.
# Positive = underdelivering. Every company, every year, in one chart.
# This is your consulting bombshell — the data that backs up Act II.

gap_col_name = [c for c in df_capex_gap.columns if 'Gap' in c and 'Pledge' in c]

if not gap_col_name:
    print("Column names in df_capex_gap:")
    print(df_capex_gap.columns.tolist())
else:
    gap_col_name = gap_col_name[0]
    yr_g  = [c for c in df_capex_gap.columns if c.strip().lower() == 'year'][0]
    co_g  = [c for c in df_capex_gap.columns if 'company' in c.lower()][0]

    gap_df = (df_capex_gap
              .dropna(subset=[co_g, yr_g, gap_col_name])
              .copy())
    gap_df = gap_df[gap_df[co_g].isin(companies)]
    gap_df[yr_g]         = gap_df[yr_g].astype(int)
    gap_df[gap_col_name] = pd.to_numeric(gap_df[gap_col_name], errors='coerce')
    gap_df = gap_df[gap_df[yr_g] >= 2020]

    fig = make_fig()

    for company, color in zip(companies, comp_colors):
        cdf = gap_df[gap_df[co_g] == company].sort_values(yr_g)
        fig.add_trace(go.Bar(
            x=cdf[yr_g].astype(str),
            y=cdf[gap_col_name],
            name=company,
            marker=dict(
                color=color,
                line=dict(color='#21262D', width=0.5)
            ),
            hovertemplate=(
                f'<b>{company}</b><br>'
                'Year: %{x}<br>'
                'Gap: %{y:.1f} %pts<br>'
                '<extra></extra>'
            )
        ))

    # Zero line — on target
    fig.add_hline(
        y=0,
        line_color='#8B949E',
        line_width=1.5,
        annotation_text='On target',
        annotation_font=dict(color='#8B949E', size=9),
        annotation_position='right'
    )

    # Red zone shading above zero
    fig.add_hrect(
        y0=0, y1=50,
        fillcolor=PALETTE['red'],
        opacity=0.05,
        layer='below',
        line_width=0
    )

    fig.update_layout(
        title=dict(
            text='Sustainability Pledge vs Reality: How Much Are Hyperscalers Missing By?',
            font=dict(size=15, color='#E6EDF3'), x=0.5
        ),
        barmode='group',
        xaxis=dict(title='Year', gridcolor='#21262D'),
        yaxis=dict(
            title='Gap: Pledged Reduction − Actual Change (%pts)',
            gridcolor='#21262D'
        ),
        height=520,
        legend=dict(
            x=1.02, y=1.0,
            xanchor='left', yanchor='top'
        ),
        annotations=[
            go.layout.Annotation(
                x=0.01, y=0.97, xref='paper', yref='paper',
                text='Above 0 = underdelivering vs pledge',
                showarrow=False,
                font=dict(size=10, color=PALETTE['red']),
                bgcolor='#21262D', borderpad=4
            ),
            go.layout.Annotation(
                text='Source: Company Reports | Carbone4 2024 | ITU/WBA 2024',
                x=0, y=-0.12, xref='paper', yref='paper',
                showarrow=False,
                font=dict(size=9, color='#8B949E')
            )
        ]
    )

    fig.write_html(str(OUT_DIR / "02_chart6_sustainability_gap.html"))
    fig.show()
    print("✅ Chart 6 saved")

✅ Chart 6 saved


In [23]:
# ── What this proves ─────────────────────────────────────────────────────────
# Which AI KPI is the strongest predictor of DC energy demand?
# The answer directly determines which features we prioritise
# in the regression model in Step 3.

corr_exclude = ['Year', flag_col]
corr_cols    = [c for c in df_hist.columns if c not in corr_exclude]

# Merge training compute index from AI KPIs
compute_col = [c for c in df_kpis_hist.columns if 'Compute' in c and 'Index' in c]
merged      = df_hist.copy()
merged['Year'] = merged['Year'].astype(int)

if compute_col:
    kpis_m = df_kpis_hist[['Year', compute_col[0]]].copy()
    kpis_m['Year'] = kpis_m['Year'].astype(int)
    merged = merged.merge(kpis_m, on='Year', how='left')
    corr_cols.append(compute_col[0])

corr_data   = merged[corr_cols].apply(pd.to_numeric, errors='coerce')
corr_matrix = corr_data.corr(method='pearson')

rename_map = {
    'Global DC Energy (TWh)':             'DC Energy (TWh)',
    'AI Share of DC Energy (%)':          'AI Energy Share %',
    'Hyperscaler Electricity (TWh)':      'Hyperscaler TWh',
    'Global AI Private Inv. ($B)':        'AI Investment $B',
    'GenAI Investment ($B)':              'GenAI Inv $B',
    'GPU/AI Chip Revenue ($B)':           'GPU Revenue $B',
    'GPU Shipments for AI (M units)':     'GPU Shipments M',
    'Hyperscale DC Count':                'Hyperscale Count',
    'DC Carbon Emissions (MtCO2e)':       'DC Carbon MtCO2e',
    'Renewable % of DC Electricity':      'Renewable %',
    'Global DC Capex ($B)':               'DC Capex $B',
}
if compute_col:
    rename_map[compute_col[0]] = 'Training Compute Index'

corr_matrix = corr_matrix.rename(index=rename_map, columns=rename_map)

# Lower triangle only — upper triangle set to None
mask     = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
plot_mat = corr_matrix.copy().astype(float)
plot_mat[mask] = None

cols = list(corr_matrix.columns)

fig = make_fig()

fig.add_trace(go.Heatmap(
    z=plot_mat.values,
    x=cols,
    y=cols,
    colorscale='RdYlGn',
    zmid=0, zmin=-1, zmax=1,
    text=np.where(mask, '', plot_mat.round(2).astype(str)),
    texttemplate='%{text}',
    textfont=dict(size=10),
    hoverongaps=False,
    colorbar=dict(title='Pearson r', thickness=14),
))

fig.update_layout(
    title=dict(
        text='Correlation Matrix: AI Boom KPIs vs Data Center Metrics',
        font=dict(size=16, color='#E6EDF3'), x=0.5
    ),
    xaxis=dict(tickangle=-45, gridcolor='#21262D'),
    yaxis=dict(gridcolor='#21262D', autorange='reversed'),
    height=650,
    margin=dict(b=120),
)

fig.write_html(str(OUT_DIR / "02_chart7_correlation_heatmap.html"))
fig.show()

# Print ranked correlations with DC Energy
print("\n📊 Correlations with DC Energy (TWh) — Feature Selection for Step 3:")
print("-" * 58)
dc_col   = [c for c in corr_matrix.columns if 'DC Energy' in c][0]
dc_corrs = (corr_matrix[dc_col]
            .drop(dc_col)
            .dropna()
            .sort_values(ascending=False))

for feat, val in dc_corrs.items():
    bar = '█' * int(abs(val) * 20)
    direction = '↑' if val > 0 else '↓'
    print(f"  {direction} {feat:<35} {val:+.3f}  {bar}")

print("\n✅ Chart 7 saved")


📊 Correlations with DC Energy (TWh) — Feature Selection for Step 3:
----------------------------------------------------------
  ↑ Hyperscaler TWh                     +0.988  ███████████████████
  ↑ Hyperscale Count                    +0.982  ███████████████████
  ↑ AI Energy Share %                   +0.978  ███████████████████
  ↑ GenAI Inv $B                        +0.973  ███████████████████
  ↑ DC Carbon MtCO2e                    +0.971  ███████████████████
  ↑ DC Capex $B                         +0.969  ███████████████████
  ↑ GPU Revenue $B                      +0.960  ███████████████████
  ↑ GPU Shipments M                     +0.960  ███████████████████
  ↑ Renewable %                         +0.958  ███████████████████
  ↑ Training Compute Index              +0.911  ██████████████████
  ↑ AI Investment $B                    +0.867  █████████████████

✅ Chart 7 saved


In [24]:
print("=" * 60)
print("✅ STEP 2 COMPLETE — All 7 Charts Built")
print("=" * 60)
print("""
Saved to outputs/charts/:
  02_chart1_dc_energy_hero.html
  02_chart2_ai_boom_overlay.html
  02_chart3_compute_scaling_law.html
  02_chart4_inference_energy.html
  02_chart5_hyperscaler_paradox.html
  02_chart6_sustainability_gap.html
  02_chart7_correlation_heatmap.html

Key findings confirmed:
  → DC energy grew 114% from 2015 to 2024
  → All 11 AI KPIs correlate above 0.86 with DC energy
  → Hyperscaler TWh is the strongest single predictor at 0.988
  → AI Energy Share % at 0.978 directly links AI boom to energy
  → Renewable % rising alongside emissions confirms the paradox
  → Power-law slope 0.861 — doubling model size = 1.82x energy

Next → 03_regression.ipynb
""")

✅ STEP 2 COMPLETE — All 7 Charts Built

Saved to outputs/charts/:
  02_chart1_dc_energy_hero.html
  02_chart2_ai_boom_overlay.html
  02_chart3_compute_scaling_law.html
  02_chart4_inference_energy.html
  02_chart5_hyperscaler_paradox.html
  02_chart6_sustainability_gap.html
  02_chart7_correlation_heatmap.html

Key findings confirmed:
  → DC energy grew 114% from 2015 to 2024
  → All 11 AI KPIs correlate above 0.86 with DC energy
  → Hyperscaler TWh is the strongest single predictor at 0.988
  → AI Energy Share % at 0.978 directly links AI boom to energy
  → Renewable % rising alongside emissions confirms the paradox
  → Power-law slope 0.861 — doubling model size = 1.82x energy

Next → 03_regression.ipynb

